In [8]:
import pandas as pd
import json
import os
import sys
import gc
import re
import torch
import accelerate
import transformers
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download
import ast

!pwd
%cd decks

/home/charlie/python/ipynb/MTG/decks
[Errno 2] No such file or directory: 'decks'
/home/charlie/python/ipynb/MTG/decks


In [9]:
current_path = os.getcwd()
print(f"📍 Current Directory: {current_path}")

📍 Current Directory: /home/charlie/python/ipynb/MTG/decks


In [10]:
# !pip uninstall torch torchvision torchaudio -y
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 --force-reinstall
# !pip install --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# !pip install --no-cache-dir transformers==4.46.0 accelerate==0.34.0

In [11]:
# ==========================================
# CHECKPOINT 0: Environment & Version Check
# ==========================================
print(f"--- 🛠️ Environment Diagnostic ---")
print(f"Python:       {sys.version.split(' ')[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Accelerate:   {accelerate.__version__}")

# GPU Verification
cuda_ready = torch.cuda.is_available()
print(f"CUDA Available: {cuda_ready}")
if cuda_ready:
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
    # Clear cache to start fresh on your 2080 Super
    torch.cuda.empty_cache()
else:
    print("⚠️ WARNING: GPU not detected. Script will be extremely slow.") 

# Force Python to find unreferenced objects and PyTorch to release VRAM
gc.collect()
torch.cuda.empty_cache()
print(f"🧹 GPU Cache Cleared. VRAM Reserved: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")

--- 🛠️ Environment Diagnostic ---
Python:       3.11.9
PyTorch:      2.7.0+cu126
Transformers: 4.39.3
Accelerate:   1.13.0
CUDA Available: True
Device Name:    NVIDIA RTX A4500
🧹 GPU Cache Cleared. VRAM Reserved: 7290.00 MB


In [12]:
# ==========================================
# CHECKPOINT 1: Path & Model Config
# ==========================================
# ONLINE_ID = "valhalla/distilbart-mnli-12-3"
# LOCAL_PATH = os.path.abspath("./local_model/distilbart-mnli-12-3")
ONLINE_ID = "microsoft/Phi-3-mini-4k-instruct"
LOCAL_PATH = os.path.abspath("../local_model/phi3_model")

print(f"\n--- 📂 Model Path Check ---")
print(f"Local Path: {LOCAL_PATH}")

# Ensure local directory exists or download
if not os.path.exists(LOCAL_PATH):
    print("📥 Local model not found. Downloading...")
    snapshot_download(
        repo_id=ONLINE_ID,
        local_dir=LOCAL_PATH,
        local_dir_use_symlinks=False,
        allow_patterns=["*.json", "*.safetensors", "*.model", "*.py"]
    )
else:
    print("✅ Local model directory detected.")


--- 📂 Model Path Check ---
Local Path: /home/charlie/python/ipynb/MTG/local_model/phi3_model
✅ Local model directory detected.


In [13]:
# ==========================================
# CHECKPOINT 2: Model & Tokenizer Loading
# ==========================================
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import os

LOCAL_PATH = "/home/charlie/python/ipynb/MTG/local_model/phi3_model"

print(f"⌛ Loading from: {LOCAL_PATH}")

try:
    tokenizer = AutoTokenizer.from_pretrained(
        LOCAL_PATH, 
        local_files_only=True, 
        trust_remote_code=True
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        LOCAL_PATH,
        torch_dtype="auto", 
        device_map="auto", 
        local_files_only=True,
        trust_remote_code=True
    )
    
    classifier = pipeline("text-generation", model=model, tokenizer=tokenizer)
    print(f"✅ Success! Pipeline active on: {next(model.parameters()).device}")
    print(f"Current VRAM Reserved: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")

except Exception as e:
    print(f"❌ Load Failed: {e}")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


⌛ Loading from: /home/charlie/python/ipynb/MTG/local_model/phi3_model


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
The model 'Phi3ForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 'MptForCausalLM', 'Musi

✅ Success! Pipeline active on: cuda:0
Current VRAM Reserved: 9638.00 MB


In [14]:
# ==========================================
# CHECKPOINT 3: Manual Feature Extraction
# ==========================================
def load_taxonomy(file_path="/home/charlie/python/ipynb/MTG/mtg_keywords.csv"):
    if not os.path.exists(file_path):
        print(f"⚠️ Error: {file_path} not found.")
        return []
        
    df = pd.read_csv(file_path).dropna(subset=['keyword', 'tag_name'])
    taxonomy = []
    
    for _, row in df.iterrows():
        raw_kw = str(row['keyword']).lower().strip()
        tag = str(row['tag_name']).lower().strip()
        
        # Split by asterisk to handle wildcards safely
        # We escape each part and join with .*? (non-greedy match anything)
        parts = [re.escape(p.strip()) for p in raw_kw.split('*') if p.strip()]
        if parts:
            pattern = ".*?".join(parts)
            taxonomy.append((pattern, tag))
        
    return taxonomy

HARD_TAXONOMY_MAP = load_taxonomy()

if len(HARD_TAXONOMY_MAP) > 0:
    print(f"✅ Success! Loaded {len(HARD_TAXONOMY_MAP)} keyword pairs.")
    print(f"Example Pair: {HARD_TAXONOMY_MAP[0]}") 
else:
    print("❌ Failed to load taxonomy. Check if mtg_keywords.csv exists.")

def extract_manual_features(name, type_line, text):
    text = str(text) if text else ""
    text_lower = text.lower()
    
    # 1. Improved Mana Production Detection
    # This regex looks for "Add" followed by any amount of mana symbols like {B}{B}{B}
    mana_match = re.search(r"[Aa]dd\s+((?:\{[WUBRGC]\}|\{[0-9]+\})+)", text)
    
    if mana_match:
        # This will capture "{B}{B}{B}" from Dark Ritual 
        # and "{R}{R}{R}{R}{R}" from Seething Song automatically.
        mana_prod = mana_match.group(1)
    elif "add one mana of any color" in text_lower:
        mana_prod = "{Any}"
    else:
        mana_prod = "0"

    # 2. Card Draw (Numerical)
    draw_count = 0
    num_map = {"two": 2, "three": 3, "four": 4, "five": 5}
    draw_match = re.search(r"draw (\d+|two|three|four|five) card", text_lower)
    if draw_match:
        val = draw_match.group(1)
        draw_count = int(val) if val.isdigit() else num_map.get(val, 1)
    elif "draw a card" in text_lower:
        draw_count = 1

    # 3. Manual Tags (Keyword Taxonomy)
    manual_tags = []
    for pattern, tag_name in HARD_TAXONOMY_MAP:
        if re.search(pattern, text_lower, re.IGNORECASE):
            manual_tags.append(tag_name)

    return mana_prod, draw_count, manual_tags

✅ Success! Loaded 57 keyword pairs.
Example Pair: ('add.*?to\\ your\\ mana\\ pool\\.', 'mana-production')


In [15]:
# ==========================================
# CHECKPOINT 4: Hybrid Reasoning Loop
# ==========================================
VISIBLE_CP = 'trunc_mtg_classification_checkpoint.csv'
df = pd.read_csv('trunc_mtg_enriched_data_master_FINAL.csv')
unique_cards = df[['name', 'type_line', 'effects_line']].drop_duplicates()
results = {}

# 1. Resume Logic
if os.path.exists(VISIBLE_CP):
    cp_df = pd.read_csv(VISIBLE_CP)
    for _, row in cp_df.drop_duplicates(subset=['name']).iterrows():
        def safe_list(v): 
            try: return ast.literal_eval(str(v)) if isinstance(v, str) else v
            except: return []
        results[row['name']] = {
            'mana_production': str(row.get('mana_production', '0')),
            'card_draw': int(row.get('card_draw', 0)),
            'manual_tags': safe_list(row.get('manual_tags')),
            'gen_tags': safe_list(row.get('generated_effect_tags'))
        }

# 2. Processing Setup
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

cards_to_process = unique_cards[~unique_cards['name'].isin(results.keys())]
cards_list = cards_to_process.to_dict('records')

# Constants
BATCH_SIZE = 8  
total_cards = len(unique_cards)

print(f"🚀 Starting Processing: {len(cards_list)} cards remaining (Total unique: {total_cards})")

for i in range(0, len(cards_list), BATCH_SIZE):
    batch = cards_list[i : i + BATCH_SIZE]
    batch_prompts = []
    batch_metadata = []

    for row in batch:
        m_mana, m_draw, m_tags = extract_manual_features(row['name'], row['type_line'], row['effects_line'])
        
        messages = [{"role": "user", "content": (
            f"Identify 2 strategic MTG archetypes for this card.\n"
            f"Card: {row['name']}\nText: {row['effects_line']}\n"
            f"Keywords: {', '.join(m_tags) if m_tags else 'None'}\n"
            f"Return ONLY a JSON object: {{\"gen_tags\": [\"archetype1\", \"archetype2\"]}}"
        )}]
        
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        batch_prompts.append(prompt)
        
        batch_metadata.append({
            'name': row['name'], 'm_mana': m_mana, 'm_draw': m_draw, 'm_tags': m_tags
        })

    # Inference
    outputs = classifier(batch_prompts, max_new_tokens=128, batch_size=BATCH_SIZE, 
                         do_sample=True, temperature=0.1, return_full_text=False)

    # Parsing
    for metadata, res in zip(batch_metadata, outputs):
        raw_output = res[0]['generated_text'].strip()
        g_tags = []
        try:
            json_match = re.search(r"\{.*\}", raw_output, re.DOTALL)
            if json_match:
                g_tags = json.loads(json_match.group(0).replace("'", '"')).get('gen_tags', [])
            else:
                g_tags = re.findall(r'"([^"]*)"', raw_output)[:2]
        except:
            g_tags = []

        results[metadata['name']] = {
            'mana_production': metadata['m_mana'], 
            'card_draw': metadata['m_draw'], 
            'manual_tags': metadata['m_tags'], 
            'gen_tags': g_tags
        }

    # Normal Print Progress
    current_processed = len(results)
    percent = (current_processed / total_cards) * 100
    print(f"📦 [{current_processed}/{total_cards}] {percent:.1f}% complete... (Last: {batch[-1]['name']})")
    
    # Batch Save (Every 5 batches)
    if (i // BATCH_SIZE) % 5 == 0:
        checkpoint_df = pd.DataFrame.from_dict(results, orient='index').reset_index()
        checkpoint_df.columns = ['name', 'mana_production', 'card_draw', 'manual_tags', 'generated_effect_tags']
        checkpoint_df.to_csv(VISIBLE_CP, index=False)
        torch.cuda.empty_cache()

# Final Export
df['mana_production'] = df['name'].map(lambda x: results.get(x, {}).get('mana_production', '0'))
df['card_draw'] = df['name'].map(lambda x: results.get(x, {}).get('card_draw', 0))
df['manual_tags'] = df['name'].map(lambda x: results.get(x, {}).get('manual_tags', []))
df['generated_effect_tags'] = df['name'].map(lambda x: results.get(x, {}).get('gen_tags', []))
df.to_csv('trunc_mtg_final_classified_deck.csv', index=False)
print("✅ All cards processed and saved to mtg_final_classified_deck.csv")

🚀 Starting Processing: 399 cards remaining (Total unique: 399)


You are not running the flash-attention implementation, expect numerical differences.


📦 [8/399] 2.0% complete... (Last: Brain Freeze)
📦 [16/399] 4.0% complete... (Last: Commandeer)
📦 [24/399] 6.0% complete... (Last: Fellwar Stone)
📦 [32/399] 8.0% complete... (Last: Gamble)
📦 [40/399] 10.0% complete... (Last: Izzet Signet)
📦 [48/399] 12.0% complete... (Last: March of Swirling Mist)
📦 [56/399] 14.0% complete... (Last: Misty Rainforest)
📦 [64/399] 16.0% complete... (Last: Pact of Negation)
📦 [72/399] 18.0% complete... (Last: Rhystic Study)
📦 [80/399] 20.1% complete... (Last: Snap)


--- Logging error ---
Traceback (most recent call last):
  File "/shared/apps/anaconda3/2024.02/lib/python3.11/logging/__init__.py", line 1110, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/shared/apps/anaconda3/2024.02/lib/python3.11/logging/__init__.py", line 953, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/shared/apps/anaconda3/2024.02/lib/python3.11/logging/__init__.py", line 687, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/shared/apps/anaconda3/2024.02/lib/python3.11/logging/__init__.py", line 377, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "/shared/apps/anaconda3/2024.02/lib/python3.11/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/shared/apps/anaconda3/2024.02/lib/python3.11/runpy.py", line 88, in _run

📦 [88/399] 22.1% complete... (Last: Strike It Rich)
📦 [96/399] 24.1% complete... (Last: Twitch)
📦 [104/399] 26.1% complete... (Last: Baral, Chief of Compliance)
📦 [112/399] 28.1% complete... (Last: Cephalid Coliseum)
📦 [120/399] 30.1% complete... (Last: Flameshot)
📦 [128/399] 32.1% complete... (Last: Go for Blood)
📦 [136/399] 34.1% complete... (Last: Kazuul's Fury)
📦 [144/399] 36.1% complete... (Last: Magus of the Bazaar)
📦 [152/399] 38.1% complete... (Last: Reliquary Tower)
📦 [160/399] 40.1% complete... (Last: Startling Development)
📦 [168/399] 42.1% complete... (Last: Thought Vessel)
📦 [176/399] 44.1% complete... (Last: Aetherflux Reservoir)
📦 [184/399] 46.1% complete... (Last: Boseiju, Who Shelters All)
📦 [192/399] 48.1% complete... (Last: Disciple of Bolas)
📦 [200/399] 50.1% complete... (Last: Glory-Bound Initiate)
📦 [208/399] 52.1% complete... (Last: Hushbringer)
📦 [216/399] 54.1% complete... (Last: Luxury Suite)
📦 [224/399] 56.1% complete... (Last: Plains)
📦 [232/399] 58.1% compl

In [16]:
# # --- 🛑 THE KILL SWITCH ---
# # Remove this block once you have your GPU environment set up correctly
# if not next(model.parameters()).is_cuda:
#     print("\n🛑 STOPPING RUN: Model is on CPU.")
#     print("Classification on CPU will be extremely slow for a full deck.")
#     print("Please request a GPU (srun/sbatch) or run this on your 2080 Super.")
#     sys.exit("Execution halted to save time.")

In [17]:
# ==========================================
# FINAL STEP: Hide Checkpoint (Archival Only)
# ==========================================
VISIBLE_CP = 'trunc_mtg_classification_checkpoint.csv'
HIDDEN_CP = '.trunc_mtg_classification_checkpoint.csv'

if os.path.exists(VISIBLE_CP):
    # Remove previous hidden archive if it exists
    if os.path.exists(HIDDEN_CP):
        os.remove(HIDDEN_CP)
    
    os.rename(VISIBLE_CP, HIDDEN_CP)
    print(f"✅ Run complete. Checkpoint archived as hidden file: {HIDDEN_CP}")
else:
    print("ℹ️ No visible checkpoint found to rename.")

✅ Run complete. Checkpoint archived as hidden file: .trunc_mtg_classification_checkpoint.csv
